# SentinelDesk triage classifier: training
Calibrated ensemble of TF-IDF LogReg + char-ngram kNN + multilingual-embedding kNN. Predicts `type`, `queue`, `priority` (EN/DE).
Uses only the 80% modelling split (90% train / 10% val). The 20% test split is never loaded.

## 1. Setup

In [ ]:
import os, sys, time
from pathlib import Path

os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")  # lower RAM use

ROOT = Path.cwd()  # move to the sentineldesk root
while not (ROOT / "src" / "sentinel_classifier.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook inside sentineldesk/")
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, classification_report

from src.sentinel_classifier import (TARGETS, RANDOM_STATE, KNN_SOURCES, TriageModel, apply_temperature,
                                     build_text, clean_text, embed_texts, fit_temperature, _weight_grid)
from src.lang import detect

CSV_PATH = ROOT / "data" / "raw" / "tickets.csv"
OUT_DIR = ROOT / "artifacts" / "sentinel_model"
KNN_K = 10
TARGET_HPI_RECALL = 0.90   # min recall for high-priority incidents
TARGET_AUTO_ACC = 0.80     # min queue accuracy for auto-routed tickets
SAVE = False               # True overwrites artifacts/sentinel_model
pd.set_option("display.precision", 3)

## 2. Data and split (test discarded)

In [ ]:
df = pd.read_csv(CSV_PATH)
for c in ["subject", "body", "answer"]:
    df[c] = df[c].map(clean_text)
df["text"] = [build_text(s, b) for s, b in zip(df["subject"], df["body"])]
df = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
df["ticket_id"] = [f"TKT-{i:05d}" for i in range(len(df))]

n_total = len(df)
kb_df = df.iloc[:int(n_total * 0.8)].copy().reset_index(drop=True)
del df  # test split: never used

is_val = np.random.RandomState(RANDOM_STATE).rand(len(kb_df)) < 0.10
tr, va = kb_df[~is_val].copy(), kb_df[is_val].copy()
print(f"dataset={n_total} | modelling={len(kb_df)} (train={len(tr)}, val={len(va)}) | test={n_total-len(kb_df)} (unused)")

## 3. Quick look

In [ ]:
display(pd.concat({t: kb_df[t].value_counts(normalize=True) for t in TARGETS}).rename("share").to_frame())
print("language label:", kb_df.language.value_counts().to_dict())

`language` label is noisy: some `de` tickets are English text. Per-language metrics use the detected language.

In [ ]:
sample_de = kb_df[kb_df.language == "de"].sample(1000, random_state=0)
print("'de'-labelled tickets written in English:", f"{np.mean([detect(t) == 'en' for t in sample_de.text]):.1%}")

## 4. Multilingual embeddings (cached if available)

In [ ]:
t0 = time.time()
cache, cached_kb = OUT_DIR / "kb_emb.npy", OUT_DIR / "kb.parquet"
if cache.exists() and cached_kb.exists() and \
        (pd.read_parquet(cached_kb, columns=["ticket_id"]).ticket_id.values == kb_df.ticket_id.values).all():
    kb_emb = np.load(cache)
    print("cached:", cache)
else:
    kb_emb = embed_texts(kb_df["text"].tolist())
    print(f"computed in {time.time()-t0:.0f}s")
print("shape:", kb_emb.shape)

## 5. Engine 1: TF-IDF + Logistic Regression (train only)

In [ ]:
t0 = time.time()
classes = {t: sorted(tr[t].unique()) for t in TARGETS}
yof = lambda f, t: f[t].map({c: i for i, c in enumerate(classes[t])}).values

vec_word = TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True, max_features=200_000)
vec_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True, max_features=200_000)
Xtr = hstack([vec_word.fit_transform(tr["text"]), vec_char.fit_transform(tr["text"])]).tocsr()
print("features:", Xtr.shape)

lr = {}
for t in TARGETS:
    lr[t] = LogisticRegression(max_iter=2000, C=4, class_weight="balanced").fit(Xtr, yof(tr, t))
    print(f"  {t} done ({time.time()-t0:.0f}s)")
del Xtr

## 6. Engines 2-3: kNN on char TF-IDF and embeddings (index = train only)

In [ ]:
cols = ["ticket_id", "subject", "body", "answer", "text", "type", "queue", "priority", "language"]
names = ["tfidf"] + list(KNN_SOURCES)
state = {"vec_word": vec_word, "vec_char": vec_char, "lr": lr, "classes": classes, "knn_k": KNN_K,
         "knn_sources": list(KNN_SOURCES),
         "weights": {t: {n: 1 / len(names) for n in names} for t in TARGETS},
         "temps": {t: 1.0 for t in TARGETS}, "gate": {}}
m = TriageModel(state, tr[cols], kb_emb[~is_val])

t0 = time.time()
comp, knn = m.components(va["text"].tolist())
print(f"val probabilities: {time.time()-t0:.0f}s")

va_lang = np.array([detect(t) for t in va["text"]])
acc = {}
for n in names:
    for t in TARGETS:
        ok = comp[n][t].argmax(1) == yof(va, t)
        acc[(n, t)] = {"all": ok.mean(), "de": ok[va_lang == "de"].mean(), "en": ok[va_lang == "en"].mean()}
display(pd.DataFrame(acc).T)

## 7. Ensemble weights (grid, max macro-F1) and temperature scaling

In [ ]:
for t in TARGETS:
    y = yof(va, t)
    mix = lambda w: sum(w[n] * comp[n][t] for n in names)
    w = max(_weight_grid(names), key=lambda w: f1_score(y, mix(w).argmax(1), average="macro"))
    state["weights"][t] = {n: round(v, 2) for n, v in w.items()}
    state["temps"][t] = fit_temperature(mix(w), y)

display(pd.DataFrame(state["weights"]).T.assign(temperature=pd.Series(state["temps"])))
pv = {t: apply_temperature(m._blend(comp, t), state["temps"][t]) for t in TARGETS}

### Calibration check

In [ ]:
rows = []
for t in TARGETS:
    conf, ok = pv[t].max(1), pv[t].argmax(1) == yof(va, t)
    g = pd.DataFrame({"bin": pd.cut(conf, [0, .4, .5, .6, .7, .8, .9, 1.0]), "conf": conf, "ok": ok}).groupby("bin", observed=True)
    ece = (g.size() / len(conf) * (g.conf.mean() - g.ok.mean()).abs()).sum()
    rows.append(g.agg(n=("ok", "size"), confidence=("conf", "mean"), accuracy=("ok", "mean")).assign(target=t))
    print(f"{t}: ECE = {ece:.3f}")
display(pd.concat(rows).reset_index().set_index(["target", "bin"]))

## 8. Escalation gate thresholds

In [ ]:
sv = knn[m.primary][1]
inc, high = classes["type"].index("Incident"), classes["priority"].index("high")
yt = (yof(va, "type") == inc) & (yof(va, "priority") == high)
score = pv["type"][:, inc] * pv["priority"][:, high]
ths = np.linspace(0.01, 0.9, 180)
ok = [th for th in ths if recall_score(yt, score >= th) >= TARGET_HPI_RECALL]
conf_q = pv["queue"].max(1)
correct = pv["queue"].argmax(1) == yof(va, "queue")
order = np.argsort(-conf_q)
cum = np.cumsum(correct[order]) / np.arange(1, len(order) + 1)
good = np.where(cum >= TARGET_AUTO_ACC)[0]
state["gate"] = {"HPI_THRESHOLD": float(max(ok) if ok else ths[0]),
                 "CONF_THRESHOLD_QUEUE": float(conf_q[order][good[-1]]) if len(good) else 1.0,
                 "SIM_THRESHOLD": float(np.quantile(sv[:, 0], 0.05))}
hpi_pred = score >= state["gate"]["HPI_THRESHOLD"]
print(state["gate"])
print(f"HPI recall = {recall_score(yt, hpi_pred):.3f} | HPI escalation rate = {hpi_pred.mean():.1%}")
print(f"auto-routed by queue = {(conf_q >= state['gate']['CONF_THRESHOLD_QUEUE']).mean():.1%}")

## 9. Validation metrics (weights tuned on val: slightly optimistic)

In [ ]:
state["val_metrics"] = {f"{t}_{k}": round(float(v), 4) for t in TARGETS for k, v in [
    ("macro_f1", f1_score(yof(va, t), pv[t].argmax(1), average="macro")),
    ("accuracy", (pv[t].argmax(1) == yof(va, t)).mean())]}
display(pd.Series(state["val_metrics"]).to_frame("val"))

by_lang = {}
for lang in ["de", "en"]:
    mk = va_lang == lang
    by_lang[lang] = {"n": int(mk.sum()),
                     **{f"{t}_acc": (pv[t].argmax(1) == yof(va, t))[mk].mean() for t in TARGETS},
                     "conf<0.5": ((pv["type"].max(1) < .5) | (pv["priority"].max(1) < .5))[mk].mean()}
display(pd.DataFrame(by_lang))
for t in TARGETS:
    print(f"\n=== {t} ===\n" + classification_report(yof(va, t), pv[t].argmax(1), target_names=classes[t], digits=3, zero_division=0))

## 10. Previous model (LR + word kNN) vs new

In [ ]:
old_dir = ROOT / "artifacts" / "sentinel_model_v1_backup"
if old_dir.exists():
    old = TriageModel.load(str(old_dir))
    po, _, _ = TriageModel(old.s, old.kb[~is_val]).predict_proba(va["text"].tolist())
    cmp = {}
    for lang in ["de", "en"]:
        mk = va_lang == lang
        for t in TARGETS:
            y = yof(va, t)[mk]
            cmp[(lang, t)] = {"old": (po[t].argmax(1)[mk] == y).mean(), "new": (pv[t].argmax(1)[mk] == y).mean()}
    display(pd.DataFrame(cmp).T.assign(gain=lambda d: d.new - d.old))
else:
    print("no previous model backup")

## 11. Final model (kNN index = full 80% KB) and save

In [ ]:
final = TriageModel(state, kb_df[cols], kb_emb)
if SAVE:
    final.save(str(OUT_DIR))
    print("saved to", OUT_DIR)
else:
    print("SAVE=False: artifacts not overwritten")

## 12. Smoke test

In [ ]:
for s, b in [("VPN down - hospital EMR offline", "Since 7am our VPN is completely down. Doctors cannot reach the EMR."),
             ("Doppelte Abbuchung", "Hallo, ich wurde diesen Monat doppelt abgebucht. Bitte prüfen Sie meine Rechnung und die Zahlung. Danke!"),
             ("", "Do your products integrate with Salesforce? We are evaluating options for next quarter.")]:
    r = final.triage(s, b)
    print(f"{(s or b)[:45]:45s} -> {r['type']} | {r['queue']} | {r['priority']} | "
          f"conf {', '.join(f'{k} {v:.0%}' for k, v in r['confidence'].items())} | {r['latency_ms']} ms")